# Calibration — Baseline ISO 52016-1 on Apt 305

Calibrates the **unmodified** ISO 52016-1 engine on Apt 305, 50 Barry St, Carlton.

This notebook validates that the baseline engine produces consistent, reproducible results for the
reference case using a fixed weather file.

**Building:** Single 20 m² Melbourne apartment, one exposed (west) facade, five conditioned neighbours  
**Weather:** Charlton, Victoria (2.0° from central Melbourne) — bundled in the repo, nothing downloaded  
**Engine:** Unmodified ISO 52016-1 (baseline)  
**Validation:** Reproducibility check — same engine, building, and weather must always give identical results

> **Why the engine is fetched separately.** `main` on this repo currently points at the merged
> history of the modification branches, so `main`'s own `pybuildingenergy/src` is the **modified**
> engine (both window and h_ce changes applied), not the baseline. Running the engine straight off
> `main` silently calibrates the wrong thing. This notebook therefore pulls the engine source from
> `claude/pybuildingenergy-baseline-anjro8` — the byte-identical-to-upstream branch — into a separate
> folder and puts *that* first on `sys.path`. Only `apt305_building.py` (which imports no engine) comes
> from `main`.
>
> If you re-run cell 1 more than once in the same Colab session, restart the runtime first
> (Runtime > Restart session). Re-running `%cd AIB` without restarting nests the clone inside
> itself (`AIB/AIB/AIB/...`) and every path after that resolves to the wrong place.

## 1. Clone repository on main

In [ ]:
import os

REPO = "https://github.com/samiraghafarigousheh-sys/AIB.git"
MAIN_BRANCH = "main"
BASELINE_BRANCH = "claude/pybuildingenergy-baseline-anjro8"

# Private repo? Uncomment and paste a token with `repo` scope:
# TOKEN = "ghp_xxx"
# REPO = f"https://{TOKEN}@github.com/samiraghafarigousheh-sys/AIB.git"

# Always start from /content so re-running this cell can never nest the clone
# inside a previous one (AIB/AIB/AIB/...).
%cd /content
!rm -rf AIB
!git clone --quiet --branch $MAIN_BRANCH $REPO AIB
%cd /content/AIB
!git fetch --quiet origin $BASELINE_BRANCH
!git config user.email "colab@example.com"
!git config user.name "Colab"
print("cwd:", os.getcwd())
print("✓ Repository cloned on main branch")

In [ ]:
# Install dependencies (main's requirements.txt matches every branch's — no engine code in it)
!pip install -q -r pybuildingenergy/requirements.txt
print("✓ Dependencies installed")

## 2. Fetch the BASELINE engine source (not main's own, modified copy)

In [ ]:
import subprocess
from pathlib import Path

baseline_dir = Path("baseline_engine")
if baseline_dir.exists():
    import shutil
    shutil.rmtree(baseline_dir)
baseline_dir.mkdir()

# git-archive the pybuildingenergy/ tree AS IT IS on the baseline branch, and
# extract it here — this is a separate, untouched copy of the unmodified engine.
archive = subprocess.run(
    f"git archive origin/{BASELINE_BRANCH} -- pybuildingenergy | tar -x -C {baseline_dir}",
    shell=True, capture_output=True, text=True,
)
if archive.returncode != 0:
    raise RuntimeError(f"could not fetch baseline engine:\n{archive.stderr}")

baseline_src = baseline_dir / "pybuildingenergy" / "src"
assert (baseline_src / "pybuildingenergy" / "source" / "utils.py").is_file(), (
    "baseline engine did not extract where expected"
)
print(f"✓ Baseline engine extracted to: {baseline_src}")

## 3. Weather file

The Charlton EPW is committed directly in `weather_cache/` on `main` — nothing to download.

In [ ]:
import glob

epw_files = sorted(glob.glob("weather_cache/*.epw"))
assert epw_files, (
    "No EPW found in weather_cache/. This means cell 1 did not put you in the repo "
    "root — check the 'cwd:' line it printed, it must end in /content/AIB."
)
EPW = epw_files[0]
print(f"✓ Weather file: {EPW}")

## 4. Load building and weather info

In [ ]:
import sys

# Baseline engine FIRST, so `import pybuildingenergy` resolves to the unmodified
# copy in baseline_engine/, not to main's own (modified) pybuildingenergy/src.
sys.path.insert(0, str(baseline_src))
sys.path.insert(0, "examples")

from weather_melbourne import read_epw_site, site_offset_deg

lat, lon, city = read_epw_site(EPW)
offset = site_offset_deg(lat, lon)

print(f"Weather file  : {EPW}")
print(f"Station       : {city}  (lat {lat}, lon {lon})")
print(f"Offset from central Melbourne: {offset:.2f}°")
print()
print("✓ Weather pinned to single EPW — all runs use identical weather")

## 5. Run the baseline engine (first calibration run)

In [ ]:
from apt305_building import build_bui
from pybuildingenergy.source.utils import ISO52016
from pybuildingenergy.source.check_input import sanitize_and_validate_BUI
import pybuildingenergy

# Confirm we are really running the baseline copy, not main's own modified one
resolved = Path(pybuildingenergy.__file__).resolve()
assert "baseline_engine" in str(resolved), (
    f"pybuildingenergy resolved to {resolved} — expected it under baseline_engine/. "
    "Restart the runtime and re-run from cell 1."
)
print(f"Engine source : {resolved}")
print("✓ Confirmed: running the UNMODIFIED baseline engine\n")

# Build and validate
bui = build_bui()
building, _ = sanitize_and_validate_BUI(bui, fix=True)

print(f"Building: {building['building']['name']}")
print(f"Floor area: {building['building']['net_floor_area']} m²")
print(f"Adjacent zones: {building['building']['number_adj_zone']}")
print()

# Run engine
result = ISO52016.Temperature_and_Energy_needs_calculation(
    building,
    weather_source="epw",
    path_weather_file=EPW,
)
annual = result[1] if isinstance(result, tuple) else None

if annual is None:
    raise RuntimeError("Failed to get annual results")

import pandas as pd
print("✓ Baseline engine run complete")

## 6. Calibration results — baseline metrics

In [ ]:
import pandas as pd

# Extract key annual metrics
metrics = {
    "Heating need (kWh)": float(pd.to_numeric(annual["Q_H_annual_kWh"], errors="coerce").iloc[0]),
    "Cooling need (kWh)": float(pd.to_numeric(annual["Q_C_annual_kWh"], errors="coerce").iloc[0]),
    "Solar gains (kWh)": float(pd.to_numeric(annual["Q_solar_gains_kWh"], errors="coerce").iloc[0]),
    "Window transm. loss (kWh)": float(pd.to_numeric(annual["Q_tr_window_loss_kWh"], errors="coerce").iloc[0]),
    "Opaque transm. loss (kWh)": float(pd.to_numeric(annual["Q_tr_opaque_loss_kWh"], errors="coerce").iloc[0]),
    "Total transm. loss (kWh)": float(pd.to_numeric(annual["Q_tr_total_loss_kWh"], errors="coerce").iloc[0]),
}

calib_df = pd.DataFrame([
    (label, f"{value:,.2f}")
    for label, value in metrics.items()
], columns=["Metric", "Value"])

print("="*60)
print("BASELINE CALIBRATION RESULTS — Apt 305")
print("="*60)
display(calib_df)
print()
print(f"Total annual energy (heating + cooling): {metrics['Heating need (kWh)'] + metrics['Cooling need (kWh)']:,.2f} kWh")
print()
print("Expected (from README / compare_branches_apt305.py Baseline column):")
print("  Heating  85.472000 kWh")
print("  Cooling 2539.262100 kWh")

## 7. Reproducibility check — run again and compare

In [ ]:
# Run the engine a second time with identical inputs
result2 = ISO52016.Temperature_and_Energy_needs_calculation(
    building,
    weather_source="epw",
    path_weather_file=EPW,
)
annual2 = result2[1] if isinstance(result2, tuple) else None

# Extract metrics from second run
metrics2 = {
    "Heating need (kWh)": float(pd.to_numeric(annual2["Q_H_annual_kWh"], errors="coerce").iloc[0]),
    "Cooling need (kWh)": float(pd.to_numeric(annual2["Q_C_annual_kWh"], errors="coerce").iloc[0]),
    "Solar gains (kWh)": float(pd.to_numeric(annual2["Q_solar_gains_kWh"], errors="coerce").iloc[0]),
    "Window transm. loss (kWh)": float(pd.to_numeric(annual2["Q_tr_window_loss_kWh"], errors="coerce").iloc[0]),
    "Opaque transm. loss (kWh)": float(pd.to_numeric(annual2["Q_tr_opaque_loss_kWh"], errors="coerce").iloc[0]),
    "Total transm. loss (kWh)": float(pd.to_numeric(annual2["Q_tr_total_loss_kWh"], errors="coerce").iloc[0]),
}

# Compare
TOL = 1e-6  # Tolerance for identical runs
comparison = []
all_match = True

for label in metrics.keys():
    val1 = metrics[label]
    val2 = metrics2[label]
    denom = max(abs(val1), abs(val2), 1e-12)
    rel_diff = abs(val1 - val2) / denom
    match = rel_diff <= TOL
    all_match &= match
    
    comparison.append({
        "Metric": label,
        "Run 1": f"{val1:,.6f}",
        "Run 2": f"{val2:,.6f}",
        "Rel. diff": f"{rel_diff:.2e}",
        "Status": "✓" if match else "✗"
    })

comp_df = pd.DataFrame(comparison)
print("\n" + "="*80)
print("REPRODUCIBILITY TEST — Same engine, building, weather")
print("="*80)
display(comp_df)
print()
if all_match:
    print("✓ PASS  Both runs give identical results (rel. diff < 1e-6)")
    print("        Baseline calibration is reproducible and reliable.")
else:
    print("✗ FAIL  Results diverged between runs")
    print("        Check for non-determinism or numerical instability.")

## 8. Calibration validation — store baseline reference

In [ ]:
import json
from pathlib import Path

# Store calibration reference
calib_ref = {
    "engine": "ISO52016 baseline (unmodified) — claude/pybuildingenergy-baseline-anjro8",
    "building": "Apt 305, 50 Barry St, Carlton",
    "weather": Path(EPW).name,
    "weather_offset_deg": offset,
    "metrics": metrics,
    "timestamp": pd.Timestamp.now().isoformat(),
    "reproducible": all_match,
}

output_dir = Path("results/calibration_baseline")
output_dir.mkdir(parents=True, exist_ok=True)

ref_file = output_dir / "baseline_calibration_reference.json"
ref_file.write_text(json.dumps(calib_ref, indent=2), encoding="utf-8")

print(f"✓ Calibration reference saved: {ref_file}")
print()
print("This reference establishes the GOLDEN STANDARD for this building.")
print("Future runs should always match these values (to numerical precision).")

## 9. Summary

In [ ]:
print("\n" + "="*70)
print("CALIBRATION SUMMARY")
print("="*70)
print(f"\nBuilding: Apt 305 — 20 m², Melbourne, one exposed facade")
print(f"Engine:   ISO 52016-1 (unmodified baseline)")
print(f"Weather:  {Path(EPW).name} ({offset:.1f}° from central Melbourne)")
print(f"\nBaseline Results:")
print(f"  Heating:  {metrics['Heating need (kWh)']:>10,.2f} kWh/yr")
print(f"  Cooling:  {metrics['Cooling need (kWh)']:>10,.2f} kWh/yr")
print(f"  Total:    {metrics['Heating need (kWh)'] + metrics['Cooling need (kWh)']:>10,.2f} kWh/yr")
print(f"\nReproducibility: {'✓ PASS' if all_match else '✗ FAIL'}")
print(f"\nThis calibration establishes a fixed reference point for:")
print(f"  • Validating future engine modifications")
print(f"  • Ensuring consistent results on repeated runs")
print(f"  • Comparing against EnergyPlus and other benchmarks")
print("\n" + "="*70)